In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
features_df = pd.read_csv(ROOT / "data" / "features" / "final_selected_attributes.csv")
features = features_df["feature"].tolist()
features[:3]

['B_NT - proBNP', 'B_Kreatynina', 'E_diastarea']

In [13]:
import re
df = pd.read_csv(ROOT / "data" / "processed" / "cardiac_patient_data_clean.csv")
X = df[features]
X = X.rename(columns=lambda c: re.sub(r"[^0-9a-zA-Z_]+", "_", str(c)).strip("_"))
y = df['zgon']

In [14]:
import numpy as np
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}
scoring = ["roc_auc", "average_precision", "accuracy", "f1", "recall"]
rows = []

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)


for name, model in models.items():
    cv_res = cross_validate(model, X, y, cv=rskf, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for metric in scoring:
        row[metric] = cv_res[f"test_{metric}"].mean()
        row[f"{metric}_std"] = cv_res[f"test_{metric}"].std()
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("model").sort_values("roc_auc", ascending=False)
results_df.round(3)

/home/kuba/vsprojects/u-smile_problem_classes/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kuba/vsprojects/u-smile_problem_classes/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kuba/vsprojects/u-smile_problem_classes/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kuba/vsprojects/u-smile_problem_classes/.venv/lib/pytho

,roc_auc,roc_auc_std,average_precision,average_precision_std,accuracy,accuracy_std,f1,f1_std,recall,recall_std
model,,,,,,,,,,
LogReg,0.804,0.052,0.851,0.044,0.737,0.043,0.773,0.039,0.778,0.057
RandomForest,0.798,0.054,0.839,0.052,0.731,0.049,0.775,0.037,0.804,0.045
CatBoost,0.789,0.055,0.839,0.052,0.731,0.052,0.772,0.043,0.791,0.058
SVM,0.782,0.052,0.810,0.056,0.725,0.050,0.765,0.043,0.782,0.064
LightGBM,0.763,0.051,0.816,0.052,0.698,0.044,0.741,0.040,0.754,0.060
XGBoost,0.755,0.050,0.803,0.046,0.701,0.050,0.747,0.043,0.767,0.058
